In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

#### Reading data

In [0]:
df = spark.read.format("parquet")\
        .load("abfss://bronze@databrickspipeline07.dfs.core.windows.net/orders")

#### View data

In [0]:
df.display()

#### view schema

In [0]:
df.printSchema()

#### Rename the column

In [0]:
df = df.withColumnRenamed("_rescued_data", "rescued_data")

#### Drop/Delete the column

In [0]:
df = df.drop("rescued_data")
df.display()

#### Convert the datatype of column

In [0]:
df = df.withColumn("order_date", to_timestamp(col("order_date")))
df.display()

#### Add a new column

In [0]:
df = df.withColumn("year", year(col("order_date")))
df.display()

#### window function

In [0]:
df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))))
df1.display()

In [0]:
df1 = df1.withColumn("rank_flag", rank().over(Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))))
df1.display()

In [0]:
df1 = df1.withColumn("row_flag", row_number().over(Window.partitionBy("year").orderBy(desc(col("total_amount")))))
df1.display()

#### Classes - OOPS

In [0]:
class windows:
    def dense_rank(self, df):
        return df.withColumn("flag", dense_rank().over(Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))))
    def rank(self, df):
        return df.withColumn("rank_flag", rank().over(Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))))
    def row_number(self, df):
        return df.withColumn("row_flag", row_number().over(Window.partitionBy("year").orderBy(desc(col("total_amount")))))


In [0]:
df_new = df

In [0]:
obj = windows()

In [0]:
df_result = obj.dense_rank(df_new)
df_result.display()

#### Writing data

In [0]:
df.write.format("delta")\
        .mode("overwrite")\
        .save("abfss://silver@databrickspipeline07.dfs.core.windows.net/orders")

#### Creating table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_pipeline_cata.silver.orders
USING DELTA
LOCATION 'abfss://silver@databrickspipeline07.dfs.core.windows.net/orders'

In [0]:
%sql
SELECT * FROM databricks_pipeline_cata.silver.orders